# JOILang Cloud-only Advisor Compression Analysis — A6000 Strict Notebook

This notebook is configured for **A6000_SET_A** by default.

It assumes the strict cloud-only advisor patch has been applied to the JOILang repository. The notebook verifies that the new flags and advisor prompt sections exist before running.

Primary execution mode in this notebook:

- **Cloud-only advisor compression**: local LLM generation/evaluation + GPT advisor mutation proposals only.
- Cloudless fallback compression is disabled by CLI flags.
- The final gate checks that `source="compression_fallback"` and `new_by_compression_fallback` are zero.

Analysis targets:

- dataset evaluation feedback passed into the cloud advisor prompt
- exact advisor prompt sections, including Case A/B/C compression policies
- GPT advisor model key
- accepted/rejected advisor proposals and rejection reasons
- before/after prompt, block, genome, token, and DETPass changes
- paper-ready cloud-only evidence tables and interpretation template


In [3]:
# ============================================================
# 0. TOP CONFIG
# ============================================================

from pathlib import Path
import os, sys, re, json, time, math, getpass, traceback, subprocess, inspect
from datetime import datetime
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except Exception:
    display = print

SERVER_PRESET_OVERRIDE = None
REPO_OVERRIDE = None
PYTHON_OVERRIDE = "/root/llm/je/bin/python"
LOCAL_MODEL_BASE_OVERRIDE = None

SERVER_CANDIDATES = [
    {
        "server": "A100_SET_B",
        "repo": Path("/root/llm/JOILang-Server"),
        "python_candidates": [Path("/root/llm/je/bin/python"), Path("/root/llm/je/bin/python3.10"), Path(sys.executable)],
        "local_model_base": Path("/root/llm/local_models"),
    },
    {
        "server": "A6000_SET_A",
        "repo": Path("/home/mgjeong/Desktop/llm/JOILang-Server"),
        "python_candidates": [Path("/home/mgjeong/miniconda3/envs/paper-gpu/bin/python"), Path("/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10"), Path(sys.executable)],
        "local_model_base": Path("/home/mgjeong/Desktop/llm/local_models"),
    },
]

def choose_server_config():
    if REPO_OVERRIDE:
        repo = Path(REPO_OVERRIDE).resolve()
        server = SERVER_PRESET_OVERRIDE or "MANUAL"
        py = Path(PYTHON_OVERRIDE).resolve() if PYTHON_OVERRIDE else Path(sys.executable).resolve()
        local_model_base = Path(LOCAL_MODEL_BASE_OVERRIDE).resolve() if LOCAL_MODEL_BASE_OVERRIDE else repo.parent / "local_models"
        return server, repo, py, local_model_base

    for cfg in SERVER_CANDIDATES:
        if cfg["repo"].exists():
            return (
                SERVER_PRESET_OVERRIDE or cfg["server"],
                cfg["repo"].resolve(),
                next((p for p in cfg["python_candidates"] if p.exists()), Path(sys.executable)).resolve(),
                (Path(LOCAL_MODEL_BASE_OVERRIDE).resolve() if LOCAL_MODEL_BASE_OVERRIDE else cfg["local_model_base"].resolve()),
            )
    raise RuntimeError("No known JOILang-Server path found. Set REPO_OVERRIDE.")

SERVER_PRESET, REPO, PYTHON, LOCAL_MODEL_BASE = choose_server_config()
VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"

PAPER_ARTIFACT_ROOT = RESULTS_ROOT / "paper_artifacts"
FIG_DIR = PAPER_ARTIFACT_ROOT / "figures"
TABLE_DIR = PAPER_ARTIFACT_ROOT / "tables"
SUMMARY_DIR = PAPER_ARTIFACT_ROOT / "summary"
for d in [FIG_DIR, TABLE_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
    "phi35_mini": "phi35_mini",
    "gemma2_9b_it": "gemma2_9b_it",
}

MODEL_LIST_3 = [
    ("7B", "qwen25_coder_7b"),
    ("8B", "llama31_8b"),
    ("14B", "qwen25_coder_14b"),
]
RUN_MODES = [("cloudless", False), ("cloud_advisor", True)]

print("=" * 120)
print("[CONFIG]")
for name, value in [
    ("SERVER_PRESET", SERVER_PRESET),
    ("REPO", REPO),
    ("VERSION_DIR", VERSION_DIR),
    ("SCRIPT", SCRIPT),
    ("PYTHON", PYTHON),
    ("LOCAL_MODEL_BASE", LOCAL_MODEL_BASE),
    ("RESULTS_ROOT", RESULTS_ROOT),
    ("PAPER_ARTIFACT_ROOT", PAPER_ARTIFACT_ROOT),
]:
    print(f"{name}: {value} exists={Path(value).exists() if isinstance(value, Path) else ''}")
print("=" * 120)

assert REPO.exists(), REPO
assert VERSION_DIR.exists(), VERSION_DIR
assert SCRIPT.exists(), SCRIPT
assert PYTHON.exists(), PYTHON
assert LOCAL_MODEL_BASE.exists(), LOCAL_MODEL_BASE

[CONFIG]
SERVER_PRESET: A100_SET_B exists=
REPO: /root/llm/JOILang-Server exists=True
VERSION_DIR: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413 exists=True
SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py exists=True
PYTHON: /root/anaconda3/bin/python3.10 exists=True
LOCAL_MODEL_BASE: /root/llm/local_models exists=True
RESULTS_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results exists=True
PAPER_ARTIFACT_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/paper_artifacts exists=True


In [4]:
# ============================================================
# Cell 2. Environment verification
# ============================================================

def run_cmd(cmd, timeout=180, check=False, env=None):
    print("\nRUN:", " ".join(map(str, cmd)))
    p = subprocess.run(
        list(map(str, cmd)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=timeout,
        check=False,
        env=env,
    )
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed rc={p.returncode}")
    return p

env_code = r"""
import sys
print("python:", sys.executable)
try:
    import torch
    print("torch:", getattr(torch, "__version__", "NO_VERSION_ATTR"))
    print("cuda:", torch.cuda.is_available())
    print("device_count:", torch.cuda.device_count())
    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        print("device0:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch_error:", repr(e))
try:
    import transformers
    print("transformers:", getattr(transformers, "__version__", "NO_VERSION_ATTR"))
except Exception as e:
    print("transformers_error:", repr(e))
"""

print("=" * 120)
print("[PYTHON ENV]")
print("=" * 120)
run_cmd([PYTHON, "-c", env_code], timeout=180, check=True)

print("=" * 120)
print("[NVIDIA-SMI]")
print("=" * 120)
run_cmd(["nvidia-smi"], timeout=60, check=False)

print("=" * 120)
print("[OPENAI API KEY]")
print("=" * 120)
print("OPENAI_API_KEY exists:", bool(os.environ.get("OPENAI_API_KEY", "").strip()))

print("=" * 120)
print("[MODEL PATH CHECK]")
print("=" * 120)
rows=[]
for model_key, dirname in MODEL_DIRS.items():
    p = LOCAL_MODEL_BASE / dirname
    rows.append({
        "model_key": model_key,
        "path": str(p),
        "exists": p.exists(),
        "config": (p/"config.json").exists(),
        "tokenizer": (p/"tokenizer.json").exists(),
        "index": (p/"model.safetensors.index.json").exists(),
        "safetensors_count": len(list(p.glob("*.safetensors"))) if p.exists() else 0,
    })
display(pd.DataFrame(rows))

[PYTHON ENV]

RUN: /root/anaconda3/bin/python3.10 -c 
import sys
print("python:", sys.executable)
try:
    import torch
    print("torch:", getattr(torch, "__version__", "NO_VERSION_ATTR"))
    print("cuda:", torch.cuda.is_available())
    print("device_count:", torch.cuda.device_count())
    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        print("device0:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch_error:", repr(e))
try:
    import transformers
    print("transformers:", getattr(transformers, "__version__", "NO_VERSION_ATTR"))
except Exception as e:
    print("transformers_error:", repr(e))

python: /root/anaconda3/bin/python3.10
torch: NO_VERSION_ATTR
torch_error: AttributeError("module 'torch' has no attribute 'cuda'")
transformers: 4.30.2

[NVIDIA-SMI]

RUN: nvidia-smi
Fri Jun 12 11:15:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driv

,model_key,path,exists,config,tokenizer,index,safetensors_count
0,qwen25_coder_7b,/root/llm/local_models/qwen25_coder_7b,True,True,True,True,4
1,llama31_8b,/root/llm/local_models/llama31_8b,True,True,True,True,4
2,qwen25_coder_14b,/root/llm/local_models/qwen25_coder_14b,True,True,True,True,6
3,phi35_mini,/root/llm/local_models/phi35_mini,True,True,True,True,2
4,gemma2_9b_it,/root/llm/local_models/gemma2_9b_it,True,True,True,True,4


In [5]:
# ============================================================
# Cell 3. Source/flag verification
# ============================================================

REQUIRED_CLOUD_ONLY_FLAGS = [
    "--cloud-only-advisor-compression",
    "--disable-compression-fallback",
    "--advisor-proposal-k",
    "--advisor-min-usable-compression-proposals",
    "--advisor-repair-invalid-proposals",
    "--advisor-repair-max-attempts",
    "--advisor-require-block-proposal-after-detpass",
    "--advisor-require-token-delta",
    "--advisor-disallow-genome-only-after-detpass",
    "--advisor-cloud-only-strict-schema",
    "--advisor-before-after-report",
    "--advisor-include-dataset-feedback",
    "--advisor-feedback-topk",
    "--advisor-feedback-max-failures-per-family",
    "--advisor-include-prompt-before-after-context",
    "--advisor-include-case-abc-sections",
    "--advisor-write-debug-prompt-sections",
]

REQUIRED_SECTIONS = [
    "DATASET_EVALUATION_FEEDBACK",
    "CASE_A_MICRO_COMPRESSION",
    "CASE_B_THRESHOLD_BLOCK_COMPRESSION",
    "CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION",
    "STRICT_JSON_RESPONSE_SCHEMA",
    "PROMPT_TOKEN_BREAKDOWN",
    "BLOCK_TOKEN_BREAKDOWN",
]

run_ga_src = SCRIPT.read_text(encoding="utf-8", errors="replace")
advisor_path = SCRIPTS_DIR / "advisor_feedback.py"
advisor_src = advisor_path.read_text(encoding="utf-8", errors="replace") if advisor_path.exists() else ""

help_text = ""
try:
    p = subprocess.run([str(PYTHON), str(SCRIPT), "--help"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=180)
    help_text = p.stdout
    print("help rc:", p.returncode)
except Exception as e:
    print("[WARN] help failed:", repr(e))

SUPPORTED_FLAGS = set(re.findall(r"--[A-Za-z0-9][A-Za-z0-9_-]*", help_text + "\n" + run_ga_src))
print("supported flag count:", len(SUPPORTED_FLAGS))

flag_df = pd.DataFrame([
    {"flag": f, "found_in_source_or_help": f in SUPPORTED_FLAGS or f in run_ga_src}
    for f in REQUIRED_CLOUD_ONLY_FLAGS
])
display(flag_df)

section_df = pd.DataFrame([
    {"section": s, "found": (s in advisor_src) or (s in run_ga_src)}
    for s in REQUIRED_SECTIONS
])
display(section_df)

missing_flags = flag_df[~flag_df["found_in_source_or_help"]]["flag"].tolist()
missing_sections = section_df[~section_df["found"]]["section"].tolist()

if missing_flags:
    print("[WARN] Missing cloud-only flags. Cloud-only smoke will fail until implementation is merged:", missing_flags)
if missing_sections:
    print("[WARN] Missing advisor prompt sections. Prompt observability is incomplete:", missing_sections)

if globals().get("STRICT_CLOUD_ONLY_REQUIRED", False) and (missing_flags or missing_sections):
    raise RuntimeError(
        "Strict cloud-only advisor implementation is not available in this A6000 repo. "
        "Apply cloud_only_advisor_upgrade_patch.diff, rerun compileall/tests, then rerun this notebook."
    )

print("[OK] Strict cloud-only advisor source/flag verification passed.")

help rc: 0
supported flag count: 134


,flag,found_in_source_or_help
0,--cloud-only-advisor-compression,False
1,--disable-compression-fallback,False
2,--advisor-proposal-k,False
3,--advisor-min-usable-compression-proposals,False
4,--advisor-repair-invalid-proposals,False
5,--advisor-repair-max-attempts,False
6,--advisor-require-block-proposal-after-detpass,False
7,--advisor-require-token-delta,False
8,--advisor-disallow-genome-only-after-detpass,False
9,--advisor-cloud-only-strict-schema,False


,section,found
0,DATASET_EVALUATION_FEEDBACK,False
1,CASE_A_MICRO_COMPRESSION,False
2,CASE_B_THRESHOLD_BLOCK_COMPRESSION,False
3,CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION,False
4,STRICT_JSON_RESPONSE_SCHEMA,False
5,PROMPT_TOKEN_BREAKDOWN,False
6,BLOCK_TOKEN_BREAKDOWN,False


[WARN] Missing cloud-only flags. Cloud-only smoke will fail until implementation is merged: ['--cloud-only-advisor-compression', '--disable-compression-fallback', '--advisor-proposal-k', '--advisor-min-usable-compression-proposals', '--advisor-repair-invalid-proposals', '--advisor-repair-max-attempts', '--advisor-require-block-proposal-after-detpass', '--advisor-require-token-delta', '--advisor-disallow-genome-only-after-detpass', '--advisor-cloud-only-strict-schema', '--advisor-before-after-report', '--advisor-include-dataset-feedback', '--advisor-feedback-topk', '--advisor-feedback-max-failures-per-family', '--advisor-include-prompt-before-after-context', '--advisor-include-case-abc-sections', '--advisor-write-debug-prompt-sections']
[WARN] Missing advisor prompt sections. Prompt observability is incomplete: ['DATASET_EVALUATION_FEEDBACK', 'CASE_A_MICRO_COMPRESSION', 'CASE_B_THRESHOLD_BLOCK_COMPRESSION', 'CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION', 'STRICT_JSON_RESPONSE_SCHEMA', 'PR

RuntimeError: Strict cloud-only advisor implementation is not available in this A6000 repo. Apply cloud_only_advisor_upgrade_patch.diff, rerun compileall/tests, then rerun this notebook.

In [ ]:
# ============================================================
# Cell 4. run_ga_all_categories wrapper
# ============================================================

def flag_supported(flag: str) -> bool:
    return flag in SUPPORTED_FLAGS or flag in run_ga_src

def add_value(cmd, flag, value, *, required=False):
    if value is None:
        return
    if not flag_supported(flag):
        if required:
            raise RuntimeError(f"Required unsupported flag: {flag}")
        print("[SKIP unsupported flag]", flag)
        return
    cmd.extend([flag, str(value)])

def add_bool(cmd, flag, enabled, *, required=False):
    if not enabled:
        return
    if not flag_supported(flag):
        if required:
            raise RuntimeError(f"Required unsupported flag: {flag}")
        print("[SKIP unsupported flag]", flag)
        return
    cmd.append(flag)

def model_env_name(model_key):
    p = LOCAL_MODEL_BASE / MODEL_DIRS.get(model_key, model_key)
    return str(p) if p.exists() else ""

def timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def run_ga_all_categories(
    model_key="qwen25_coder_14b",
    categories=(1,),
    limit_per_category=1,
    sample_size=1,
    validation_size=1,
    population=4,
    gens=3,
    target_detpass=90,
    base_prefix=None,
    use_advisor=False,
    full_run=True,
    progress="verbose",
    timeout_sec=1200,
    retries=0,
    advisor_trigger_mode="always",
    advisor_min_population_for_child=4,
    advisor_force_child_quota=True,
    use_mock_advisor=False,
    advisor_model_key="gpt41_mini",

    compression_detpass_threshold=90,
    aggressive_compression_after_target=True,
    allow_aggressive_compression=True,
    compression_child_quota=1,
    compression_child_ratio=0.25,
    micro_compression_child_quota=1,
    micro_compression_child_ratio=0.25,
    block_compression_child_quota=1,
    block_compression_child_ratio=0.25,
    multi_block_compression_child_quota=1,
    multi_block_compression_child_ratio=0.25,
    global_budget_compression_child_quota=0,
    advisor_compression_child_quota=1,
    advisor_prefer_compression_after_detpass=90,
    compression_token_reduction_target=0.15,
    compression_token_plateau_delta=1.0,
    enable_block_token_breakdown=True,
    enable_multi_block_compression=True,
    enable_render_budget_compression=False,
    min_compression_token_delta=50,

    cloud_only_advisor_compression=False,
    disable_compression_fallback=False,
    advisor_proposal_k=6,
    advisor_min_usable_compression_proposals=2,
    advisor_repair_invalid_proposals=True,
    advisor_repair_max_attempts=2,
    advisor_require_block_proposal_after_detpass=True,
    advisor_require_token_delta=True,
    advisor_disallow_genome_only_after_detpass=True,
    advisor_cloud_only_strict_schema=True,
    advisor_before_after_report=True,
    advisor_include_dataset_feedback=True,
    advisor_feedback_topk=5,
    advisor_feedback_max_failures_per_family=3,
    advisor_include_prompt_before_after_context=True,
    advisor_include_case_abc_sections=True,
    advisor_write_debug_prompt_sections=True,
):
    if cloud_only_advisor_compression and not use_advisor:
        raise ValueError("cloud_only_advisor_compression requires use_advisor=True")

    mode_name = "cloud_only_advisor" if cloud_only_advisor_compression else ("cloud_advisor" if use_advisor else "cloudless")
    base_prefix = base_prefix or f"smoke_{SERVER_PRESET}_{mode_name}"
    cat_label = "".join(map(str, categories))

    run_dir = RESULTS_ROOT / f"{base_prefix}_{mode_name}_cat{cat_label}_lpc{limit_per_category}_pop{population}_gens{gens}_{model_key}_{timestamp()}"
    output_root = run_dir / "ga_output"
    output_root.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(PYTHON), "-u", str(SCRIPT),
        "--profile", "version0_15",
        "--model-key", model_key,
        "--target-detpass", str(target_detpass),
        "--llm-mode", "worker",
        "--population", str(population),
        "--gens", str(gens),
        "--min-generations", str(gens),
        "--max-generations", str(gens),
        "--sample-size", str(sample_size),
        "--validation-size", str(validation_size),
        "--cheap-eval-limit", "2",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--plateau-window", "1",
        "--disruptive-max-attempts", "1",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",
        "--progress", progress,
        "--timeout-sec", str(timeout_sec),
        "--retries", str(retries),
        "--limit-per-category", str(limit_per_category),
        "--output-root", str(output_root),
    ]

    if full_run:
        cmd.append("--full-run")
    for flag in [
        "--feedback-guided-mutation",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",
    ]:
        if flag_supported(flag):
            cmd.append(flag)

    for c in categories:
        cmd.extend(["--category", str(c)])

    add_value(cmd, "--compression-detpass-threshold", compression_detpass_threshold)
    add_bool(cmd, "--aggressive-compression-after-target", aggressive_compression_after_target)
    add_bool(cmd, "--allow-aggressive-compression", allow_aggressive_compression)
    add_value(cmd, "--compression-child-quota", compression_child_quota)
    add_value(cmd, "--compression-child-ratio", compression_child_ratio)
    add_value(cmd, "--micro-compression-child-quota", micro_compression_child_quota)
    add_value(cmd, "--micro-compression-child-ratio", micro_compression_child_ratio)
    add_value(cmd, "--block-compression-child-quota", block_compression_child_quota)
    add_value(cmd, "--block-compression-child-ratio", block_compression_child_ratio)
    add_value(cmd, "--multi-block-compression-child-quota", multi_block_compression_child_quota)
    add_value(cmd, "--multi-block-compression-child-ratio", multi_block_compression_child_ratio)
    add_value(cmd, "--global-budget-compression-child-quota", global_budget_compression_child_quota)
    add_value(cmd, "--advisor-compression-child-quota", advisor_compression_child_quota)
    add_value(cmd, "--advisor-prefer-compression-after-detpass", advisor_prefer_compression_after_detpass)
    add_value(cmd, "--compression-token-reduction-target", compression_token_reduction_target)
    add_value(cmd, "--compression-token-plateau-delta", compression_token_plateau_delta)
    add_bool(cmd, "--enable-block-token-breakdown", enable_block_token_breakdown)
    add_bool(cmd, "--enable-multi-block-compression", enable_multi_block_compression)
    add_bool(cmd, "--enable-render-budget-compression", enable_render_budget_compression)
    add_value(cmd, "--min-compression-token-delta", min_compression_token_delta)

    if use_advisor:
        cmd.append("--llm-mutation-advisor")
        add_value(cmd, "--advisor-model-key", advisor_model_key)
        add_value(cmd, "--advisor-trigger-mode", advisor_trigger_mode)
        add_value(cmd, "--advisor-min-population-for-child", advisor_min_population_for_child)
        add_bool(cmd, "--advisor-force-child-quota", advisor_force_child_quota)
        add_bool(cmd, "--use-mock-advisor", use_mock_advisor)

        req = bool(cloud_only_advisor_compression)
        add_bool(cmd, "--cloud-only-advisor-compression", cloud_only_advisor_compression, required=req)
        add_bool(cmd, "--disable-compression-fallback", disable_compression_fallback, required=req)
        add_value(cmd, "--advisor-proposal-k", advisor_proposal_k, required=req)
        add_value(cmd, "--advisor-min-usable-compression-proposals", advisor_min_usable_compression_proposals, required=req)
        add_bool(cmd, "--advisor-repair-invalid-proposals", advisor_repair_invalid_proposals, required=req)
        add_value(cmd, "--advisor-repair-max-attempts", advisor_repair_max_attempts, required=req)
        add_bool(cmd, "--advisor-require-block-proposal-after-detpass", advisor_require_block_proposal_after_detpass, required=req)
        add_bool(cmd, "--advisor-require-token-delta", advisor_require_token_delta, required=req)
        add_bool(cmd, "--advisor-disallow-genome-only-after-detpass", advisor_disallow_genome_only_after_detpass, required=req)
        add_bool(cmd, "--advisor-cloud-only-strict-schema", advisor_cloud_only_strict_schema, required=req)
        add_bool(cmd, "--advisor-before-after-report", advisor_before_after_report, required=req)
        add_bool(cmd, "--advisor-include-dataset-feedback", advisor_include_dataset_feedback, required=req)
        add_value(cmd, "--advisor-feedback-topk", advisor_feedback_topk, required=req)
        add_value(cmd, "--advisor-feedback-max-failures-per-family", advisor_feedback_max_failures_per_family, required=req)
        add_bool(cmd, "--advisor-include-prompt-before-after-context", advisor_include_prompt_before_after_context, required=req)
        add_bool(cmd, "--advisor-include-case-abc-sections", advisor_include_case_abc_sections, required=req)
        add_bool(cmd, "--advisor-write-debug-prompt-sections", advisor_write_debug_prompt_sections, required=req)
    else:
        add_value(cmd, "--advisor-trigger-mode", "off")

    env = os.environ.copy()
    env["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
    model_name = model_env_name(model_key)
    if model_name:
        env["JOI_V15_LOCAL_MODEL_NAME"] = model_name
    env["JOI_V15_LOCAL_FILES_ONLY"] = "true"
    env["TRANSFORMERS_OFFLINE"] = env.get("TRANSFORMERS_OFFLINE", "1")
    env["HF_HUB_OFFLINE"] = env.get("HF_HUB_OFFLINE", "1")
    env["TOKENIZERS_PARALLELISM"] = "false"
    debug_log = Path("/tmp") / f"joi_v15_worker_debug_{model_key}_{mode_name}_{timestamp()}.log"
    env["JOI_V15_WORKER_DEBUG_LOG"] = str(debug_log)

    print("\n" + "=" * 120)
    print(f"RUN: {model_key} / {mode_name}")
    print("OUTPUT:", output_root)
    print("PYTHON:", PYTHON)
    print("MODEL_BASE:", LOCAL_MODEL_BASE)
    print("MODEL_NAME:", env.get("JOI_V15_LOCAL_MODEL_NAME", "<run_ga_search default>"))
    print("DEBUG_LOG:", debug_log)
    print("=" * 120)
    print("COMMAND:")
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 120)

    proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    rc = proc.wait()

    (output_root / "_notebook_command.txt").write_text(" ".join(shlex.quote(str(x)) for x in cmd), encoding="utf-8")
    (output_root / "_notebook_stdout_tail.txt").write_text("".join(lines[-2000:]), encoding="utf-8")

    print("\nRETURN CODE:", rc)
    print("OUTPUT:", output_root)
    if debug_log.exists():
        print("DEBUG_LOG:", debug_log)
    if rc != 0:
        raise RuntimeError(f"run_ga_all_categories failed with return code {rc}")
    return output_root

print("[OK] wrapper defined")

In [ ]:
# ============================================================
# Cell 5. Artifact readers and analysis helpers
# ============================================================

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8", errors="replace"))
    except Exception as e:
        print("[JSON READ ERROR]", path, repr(e))
        return None

def read_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        return rows
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            rows.append({"_raw": line, "_parse_error": True})
    return rows

def safe_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        print("[CSV READ ERROR]", path, repr(e))
        return pd.DataFrame()

def extract_blocks(obj):
    if obj is None:
        return []
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for k in ["blocks", "block_token_breakdown", "block_breakdown", "items", "rows", "block_rows"]:
            if isinstance(obj.get(k), list):
                return obj[k]
        rows = []
        for k, v in obj.items():
            if isinstance(v, dict):
                vv = dict(v)
                vv.setdefault("block_id", k)
                rows.append(vv)
        return rows
    return []

def classify_case_abc(row):
    level = str(row.get("compression_level", "") or "").lower()
    op = str(row.get("operator", row.get("mutation_type", row.get("mutation", ""))) or "").lower()
    target = str(row.get("target_block_id", row.get("selected_block_id", "")) or "").lower()
    ids = row.get("selected_block_ids", [])
    if isinstance(ids, str):
        try:
            ids = ast.literal_eval(ids)
        except Exception:
            ids = []
    if level in {"multi_block", "global_budget", "global", "render_budget"}:
        return "Case C"
    if "multi" in op or "global" in op or "budget" in op:
        return "Case C"
    if isinstance(ids, list) and len(ids) >= 2:
        return "Case C"
    if level == "block":
        return "Case B"
    if target and target not in {"genome", "none", "nan", ""}:
        return "Case B"
    if "drop_optional_block" in op or "compact_reasoning_skeleton" in op:
        return "Case B"
    if level == "micro":
        return "Case A"
    if any(k in op for k in ["candidate_strategies", "lower_output_max_tokens", "template_compress", "dedupe", "safe"]):
        return "Case A"
    if any(k in op for k in ["few_shot", "micro_rules", "compact_block_params"]):
        return "Case B"
    return "Unclassified"

def load_progress(out_dir):
    return safe_csv(Path(out_dir) / "ga_generation_progress.csv")

def load_transitions(out_dir):
    return safe_csv(Path(out_dir) / "population_transitions.csv")

def load_all_proposals(out_dir):
    out_dir = Path(out_dir)
    rows = []
    for fn in ["advisor_mutation_proposals.jsonl", "mutation_proposals.jsonl"]:
        for r in read_jsonl(out_dir / fn):
            rr = dict(r)
            rr["_file"] = fn
            if "operator" not in rr:
                rr["operator"] = rr.get("mutation_type", rr.get("mutation", ""))
            rr["case_label_inferred"] = classify_case_abc(rr)
            rows.append(rr)
    return pd.DataFrame(rows)

def summarize_run(out_dir, label=None, mode=None):
    out_dir = Path(out_dir)
    row = {
        "label": label or out_dir.parent.name,
        "mode": mode,
        "out_dir": str(out_dir),
        "exists": out_dir.exists(),
        "summary_exists": (out_dir/"ga_summary.json").exists(),
        "progress_exists": (out_dir/"ga_generation_progress.csv").exists(),
        "transitions_exists": (out_dir/"population_transitions.csv").exists(),
        "block_breakdown_exists": (out_dir/"block_token_breakdown.json").exists(),
        "prompt_breakdown_exists": (out_dir/"prompt_token_breakdown.json").exists(),
    }
    s = read_json(out_dir/"ga_summary.json") or {}
    row["best_DETPass"] = s.get("best_DETPass", s.get("best_so_far_DETPass", np.nan))
    row["advisor_model_key"] = s.get("advisor_model_key")
    p = load_progress(out_dir)
    if len(p):
        for c in ["validation_det_pass_rate", "validation_avg_det_score", "best_so_far_DETPass"]:
            if c in p.columns:
                row[c] = float(pd.to_numeric(p[c], errors="coerce").fillna(0).max())
        if "avg_prompt_tokens" in p.columns:
            toks = pd.to_numeric(p["avg_prompt_tokens"], errors="coerce").dropna()
            if len(toks):
                row["first_tokens"] = float(toks.iloc[0])
                row["last_tokens"] = float(toks.iloc[-1])
                row["min_tokens"] = float(toks.min())
                row["token_delta_last"] = row["last_tokens"] - row["first_tokens"]
                row["token_reduction_ratio_min"] = (row["first_tokens"] - row["min_tokens"]) / row["first_tokens"] if row["first_tokens"] else np.nan
    t = load_transitions(out_dir)
    for col in [
        "new_by_compression", "new_by_micro_compression", "new_by_block_compression",
        "new_by_multi_block_compression", "new_by_global_budget_compression",
        "new_by_compression_fallback", "new_by_advisor",
        "advisor_proposals_generated", "advisor_proposals_accepted_applied",
        "advisor_proposals_rejected", "advisor_unfilled_quota",
    ]:
        if len(t) and col in t.columns:
            row[col + "_sum"] = float(pd.to_numeric(t[col], errors="coerce").fillna(0).sum())
    prop = load_all_proposals(out_dir)
    row["proposal_rows"] = len(prop)
    row["fallback_rows"] = int((prop.get("source", pd.Series([], dtype=str)).astype(str) == "compression_fallback").sum()) if len(prop) else 0
    row["advisor_accepted_rows"] = int(((prop.get("_file", pd.Series([], dtype=str)) == "advisor_mutation_proposals.jsonl") & (prop.get("accepted", pd.Series([], dtype=bool)) == True)).sum()) if len(prop) else 0
    row["advisor_rejected_rows"] = int(((prop.get("_file", pd.Series([], dtype=str)) == "advisor_mutation_proposals.jsonl") & (prop.get("accepted", pd.Series([], dtype=bool)) == False)).sum()) if len(prop) else 0
    return row

def prompt_keyword_table(out_dir):
    out_dir = Path(out_dir)
    keys = [
        "ADVISOR_ROLE", "CURRENT_STATE", "DATASET_EVALUATION_FEEDBACK",
        "PROMPT_TOKEN_BREAKDOWN", "BLOCK_TOKEN_BREAKDOWN",
        "CASE_A_MICRO_COMPRESSION", "CASE_B_THRESHOLD_BLOCK_COMPRESSION",
        "CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION", "STRICT_JSON_RESPONSE_SCHEMA",
    ]
    rows = []
    for p in sorted(out_dir.glob("advisor_prompt_generation_*.txt")):
        txt = p.read_text(encoding="utf-8", errors="replace")
        row = {"file": p.name, "chars": len(txt), "approx_tokens": round(len(txt)/4)}
        for k in keys:
            row[k] = k in txt
        rows.append(row)
    return pd.DataFrame(rows)

def inspect_run(out_dir, max_rows=30):
    out_dir = Path(out_dir)
    print("\n" + "#"*120)
    print("INSPECT RUN:", out_dir)
    print("#"*120)
    files = [
        "ga_summary.json", "ga_generation_progress.csv", "population_transitions.csv",
        "mutation_proposals.jsonl", "advisor_mutation_proposals.jsonl",
        "advisor_mutation_summary.csv", "advisor_cloud_only_summary.csv",
        "advisor_cloud_only_before_after.csv", "advisor_case_ABC_summary.csv",
        "block_token_breakdown.json", "prompt_token_breakdown.json",
    ]
    for f in files:
        print(f"{f:45s}", (out_dir/f).exists())
    print("\n[SUMMARY]")
    display(pd.DataFrame([summarize_run(out_dir)]))
    p = load_progress(out_dir)
    if len(p):
        cols = [c for c in ["generation","validation_det_pass_rate","validation_avg_det_score","best_so_far_DETPass","avg_prompt_tokens","compression_ready","compression_phase","generation_phase","next_action"] if c in p.columns]
        print("\n[GA PROGRESS]")
        display(p[cols].tail(max_rows))
    t = load_transitions(out_dir)
    if len(t):
        cols = [c for c in ["generation","fallback_disabled","new_by_compression","new_by_micro_compression","new_by_block_compression","new_by_multi_block_compression","new_by_compression_fallback","new_by_advisor","advisor_proposals_generated","advisor_proposals_accepted_applied","advisor_proposals_rejected","advisor_unfilled_quota"] if c in t.columns]
        print("\n[TRANSITIONS]")
        display(t[cols].tail(max_rows))
    print("\n[ADVISOR PROMPT KEYWORDS]")
    df = prompt_keyword_table(out_dir)
    display(df if len(df) else pd.DataFrame([{"message":"No advisor prompt files"}]))
    print("\n[PROPOSALS]")
    prop = load_all_proposals(out_dir)
    if len(prop):
        cols = [c for c in ["_file","generation","source","schema_source","case_label","case_label_inferred","operator","compression_level","selected_block_id","selected_block_ids","expected_token_delta","measured_prompt_token_delta","accepted","applied","rejection_reason","fallback_reason","dataset_feedback_reason"] if c in prop.columns]
        display(prop[cols].tail(max_rows))
    else:
        print("No proposal rows.")

print("[OK] helpers defined")

In [ ]:
# ============================================================
# Cell 6. Cloudless baseline smoke
# ============================================================

if RUN_CLOUDLESS_SMOKE:
    cloudless_out = run_ga_all_categories(
        model_key=SMOKE_MODEL_KEY,
        categories=SMOKE_CATEGORIES,
        limit_per_category=SMOKE_LIMIT_PER_CATEGORY,
        sample_size=SMOKE_SAMPLE_SIZE,
        validation_size=SMOKE_VALIDATION_SIZE,
        population=SMOKE_POPULATION,
        gens=SMOKE_GENS,
        base_prefix=f"smoke_cloudless_baseline_{SERVER_PRESET}",
        use_advisor=False,
        timeout_sec=2400,
        compression_child_quota=1,
        compression_child_ratio=0.25,
        micro_compression_child_quota=1,
        micro_compression_child_ratio=0.25,
        block_compression_child_quota=1,
        block_compression_child_ratio=0.25,
        multi_block_compression_child_quota=1,
        multi_block_compression_child_ratio=0.25,
        advisor_compression_child_quota=0,
    )
    inspect_run(cloudless_out)
else:
    print("[SKIP] RUN_CLOUDLESS_SMOKE=False. This A6000 notebook is cloud-only by default.")

In [ ]:
# ============================================================
# Cell 7. Hybrid advisor smoke
# ============================================================

if RUN_HYBRID_SMOKE:
    hybrid_out = run_ga_all_categories(
        model_key=SMOKE_MODEL_KEY,
        categories=SMOKE_CATEGORIES,
        limit_per_category=SMOKE_LIMIT_PER_CATEGORY,
        sample_size=SMOKE_SAMPLE_SIZE,
        validation_size=SMOKE_VALIDATION_SIZE,
        population=SMOKE_POPULATION,
        gens=SMOKE_GENS,
        base_prefix=f"smoke_hybrid_advisor_{SERVER_PRESET}",
        use_advisor=True,
        timeout_sec=2400,
        advisor_model_key="gpt41_mini",
        advisor_trigger_mode="always",
        advisor_min_population_for_child=4,
        advisor_force_child_quota=True,
        compression_child_quota=1,
        compression_child_ratio=0.25,
        micro_compression_child_quota=1,
        micro_compression_child_ratio=0.25,
        block_compression_child_quota=1,
        block_compression_child_ratio=0.25,
        multi_block_compression_child_quota=1,
        multi_block_compression_child_ratio=0.25,
        advisor_compression_child_quota=1,
        advisor_include_dataset_feedback=True,
        advisor_include_case_abc_sections=True,
        advisor_write_debug_prompt_sections=True,
    )
    inspect_run(hybrid_out)
else:
    print("[SKIP] RUN_HYBRID_SMOKE=False. This A6000 notebook is cloud-only by default.")

In [ ]:
# ============================================================
# Cell 8. Cloud-only advisor smoke
# Fallback disabled; advisor-only compression evidence.
# ============================================================

if RUN_CLOUD_ONLY_SMOKE:
    cloud_only_out = run_ga_all_categories(
        model_key=SMOKE_MODEL_KEY,
        categories=SMOKE_CATEGORIES,
        limit_per_category=SMOKE_LIMIT_PER_CATEGORY,
        sample_size=SMOKE_SAMPLE_SIZE,
        validation_size=SMOKE_VALIDATION_SIZE,
        population=SMOKE_POPULATION,
        gens=SMOKE_GENS,
        base_prefix=f"smoke_cloud_only_advisor_{SERVER_PRESET}",
        use_advisor=True,
        timeout_sec=2400,
        advisor_model_key="gpt41_mini",
        advisor_trigger_mode="always",
        advisor_min_population_for_child=4,
        advisor_force_child_quota=True,

        # Cloud-only isolation.
        cloud_only_advisor_compression=True,
        disable_compression_fallback=True,
        compression_child_quota=0,
        compression_child_ratio=0.0,
        micro_compression_child_quota=0,
        micro_compression_child_ratio=0.0,
        block_compression_child_quota=0,
        block_compression_child_ratio=0.0,
        multi_block_compression_child_quota=0,
        multi_block_compression_child_ratio=0.0,
        global_budget_compression_child_quota=0,
        advisor_compression_child_quota=2,

        # Strict cloud advisor analysis.
        advisor_proposal_k=6,
        advisor_min_usable_compression_proposals=2,
        advisor_repair_invalid_proposals=True,
        advisor_repair_max_attempts=2,
        advisor_require_block_proposal_after_detpass=True,
        advisor_require_token_delta=True,
        advisor_disallow_genome_only_after_detpass=True,
        advisor_cloud_only_strict_schema=True,
        advisor_before_after_report=True,
        advisor_include_dataset_feedback=True,
        advisor_feedback_topk=5,
        advisor_feedback_max_failures_per_family=3,
        advisor_include_prompt_before_after_context=True,
        advisor_include_case_abc_sections=True,
        advisor_write_debug_prompt_sections=True,
    )
    inspect_run(cloud_only_out)
else:
    raise RuntimeError("RUN_CLOUD_ONLY_SMOKE=False. This A6000 notebook is intended to run strict cloud-only smoke by default.")

In [ ]:
# ============================================================
# Cell 9. Dataset evaluation feedback inspection
# ============================================================

def select_advisor_out_dir():
    if "cloud_only_out" in globals():
        return Path(cloud_only_out)
    raise RuntimeError("No cloud_only_out found. Run Cell 8 strict cloud-only advisor smoke first.")

OUT_DIR = select_advisor_out_dir()
print("OUT_DIR:", OUT_DIR)

print("\n[advisor_prompt_case_sections_generation_*.json]")
rows = []
for p in sorted(OUT_DIR.glob("advisor_prompt_case_sections_generation_*.json")):
    obj = read_json(p) or {}
    rows.append({"file": p.name, "keys": list(obj.keys()), "has_DATASET_EVALUATION_FEEDBACK": "DATASET_EVALUATION_FEEDBACK" in obj})
display(pd.DataFrame(rows) if rows else pd.DataFrame([{"message":"No prompt case section files"}]))

print("\n[advisor_feedback_batches.jsonl]")
batches = read_jsonl(OUT_DIR/"advisor_feedback_batches.jsonl")
display(pd.DataFrame(batches).head(20) if batches else pd.DataFrame([{"message":"No advisor feedback batches"}]))

print("\n[Prompt excerpt: DATASET_EVALUATION_FEEDBACK]")
for p in sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt"))[:3]:
    txt = p.read_text(encoding="utf-8", errors="replace")
    idx = txt.find("DATASET_EVALUATION_FEEDBACK")
    print("\n" + "-"*120)
    print(p.name, "chars:", len(txt))
    if idx >= 0:
        print(txt[max(0, idx-400):idx+2600])
    else:
        print("[NOT FOUND]")

In [ ]:
# ============================================================
# Cell 10. Exact advisor prompt inspection
# ============================================================

OUT_DIR = select_advisor_out_dir()
sections = [
    "ADVISOR_ROLE", "CURRENT_STATE", "DATASET_EVALUATION_FEEDBACK",
    "PROMPT_TOKEN_BREAKDOWN", "BLOCK_TOKEN_BREAKDOWN", "PROTECTED_BLOCKS",
    "COMPRESSION_ALLOWED_BLOCKS", "CASE_A_MICRO_COMPRESSION",
    "CASE_B_THRESHOLD_BLOCK_COMPRESSION", "CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION",
    "STRICT_JSON_RESPONSE_SCHEMA",
]

def extract_section_text(text, section, max_chars=3000):
    idx = text.find(section)
    if idx < 0:
        return ""
    end = min(len(text), idx + max_chars)
    nexts = [text.find(s, idx + len(section)) for s in sections if s != section and text.find(s, idx + len(section)) > idx]
    if nexts:
        end = min(end, min(nexts))
    return text[idx:end]

prompt_rows = []
for p in sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt")):
    txt = p.read_text(encoding="utf-8", errors="replace")
    row = {"file": p.name, "chars": len(txt), "approx_tokens": round(len(txt)/4)}
    for s in sections:
        row[f"has_{s}"] = s in txt
    row["has_advisor_model_key"] = "advisor_model_key" in txt or "gpt41_mini" in txt
    prompt_rows.append(row)

prompt_summary_df = pd.DataFrame(prompt_rows)
display(prompt_summary_df)

for p in sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt"))[:2]:
    txt = p.read_text(encoding="utf-8", errors="replace")
    print("\n" + "="*120)
    print("PROMPT:", p.name, "chars:", len(txt), "approx_tokens:", round(len(txt)/4))
    for s in sections:
        sec = extract_section_text(txt, s, 2000)
        if sec:
            print("\n" + "-"*100)
            print(sec[:2000])

In [ ]:
# ============================================================
# Cell 11. Advisor raw response inspection
# ============================================================

OUT_DIR = select_advisor_out_dir()
proposal_keys = [
    "proposals", "mutation_proposals", "micro_compression_proposals",
    "block_compression_proposals", "multi_block_compression_proposals",
    "global_budget_compression_proposals",
]
resp_rows, raw_prop_rows = [], []

for p in sorted(OUT_DIR.glob("advisor_response_generation_*.json")):
    obj = read_json(p) or {}
    parsed = obj.get("parsed", {})
    raw = str(obj.get("raw_content", ""))
    m = re.search(r"generation_(\d+)", p.name)
    gen = int(m.group(1)) if m else None
    row = {
        "generation": gen, "file": p.name, "size_bytes": p.stat().st_size,
        "raw_chars": len(raw), "parsed_type": type(parsed).__name__,
        "accepted_count": len(obj.get("accepted_proposals", []) or []),
        "rejected_count": len(obj.get("rejected_proposals", []) or []),
    }
    if isinstance(parsed, dict):
        row.update({
            "advisor_status": parsed.get("advisor_status"),
            "advisor_model_key": parsed.get("advisor_model_key") or obj.get("advisor_model_key"),
            "dataset_feedback_seen": parsed.get("dataset_feedback_seen"),
            "prompt_token_breakdown_seen": parsed.get("prompt_token_breakdown_seen"),
            "block_token_breakdown_seen": parsed.get("block_token_breakdown_seen"),
        })
        for k in proposal_keys:
            v = parsed.get(k)
            row[k+"_count"] = len(v) if isinstance(v, list) else 0
            if isinstance(v, list):
                for item in v:
                    if isinstance(item, dict):
                        rr = dict(item)
                        rr["generation"] = gen
                        rr["schema_source"] = k
                        rr["_response_file"] = p.name
                        rr["operator"] = rr.get("operator", rr.get("mutation_type", rr.get("mutation", "")))
                        rr["case_label_inferred"] = classify_case_abc(rr)
                        raw_prop_rows.append(rr)
    resp_rows.append(row)

advisor_response_df = pd.DataFrame(resp_rows)
display(advisor_response_df)

raw_advisor_proposals_df = pd.DataFrame(raw_prop_rows)
print("raw advisor proposal rows:", len(raw_advisor_proposals_df))
if len(raw_advisor_proposals_df):
    cols = [c for c in ["generation","_response_file","schema_source","case_label","case_label_inferred","compression_level","operator","selected_block_id","selected_block_ids","expected_token_delta","dataset_feedback_reason","reason"] if c in raw_advisor_proposals_df.columns]
    display(raw_advisor_proposals_df[cols])

print("\n[Repair traces]")
repair_files = sorted(OUT_DIR.glob("advisor_proposal_repair_trace_generation_*.json"))
print("count:", len(repair_files))
for p in repair_files[:5]:
    print(p.name, p.stat().st_size, "bytes", list((read_json(p) or {}).keys()))

In [ ]:
# ============================================================
# Cell 12. Unified proposal table
# ============================================================

OUT_DIR = select_advisor_out_dir()
proposal_df = load_all_proposals(OUT_DIR)

if len(proposal_df):
    cols = [c for c in [
        "_file","generation","source","schema_source","case_label","case_label_inferred",
        "operator","compression_level","selected_block_id","selected_block_ids",
        "expected_token_delta","measured_prompt_token_delta","accepted","applied",
        "rejection_reason","fallback_reason","dataset_feedback_reason"
    ] if c in proposal_df.columns]
    display(proposal_df[cols])
    group_cols = [c for c in ["source","case_label_inferred","operator"] if c in proposal_df.columns]
    if group_cols:
        display(proposal_df.groupby(group_cols, dropna=False).size().reset_index(name="count"))
else:
    print("[WARN] no proposals found")

In [ ]:
# ============================================================
# Cell 13. Before/after block and prompt comparison
# ============================================================

OUT_DIR = select_advisor_out_dir()

before_after_path = OUT_DIR / "advisor_cloud_only_before_after.csv"
if before_after_path.exists():
    before_after_df = pd.read_csv(before_after_path)
    print("[advisor_cloud_only_before_after.csv]")
    display(before_after_df)
else:
    print("[WARN] advisor_cloud_only_before_after.csv not found")
    before_after_df = pd.DataFrame()

def flatten_block_diff(obj):
    rows = []
    base = {k: obj.get(k) for k in ["generation","genome_id","parent","child","proposal_id"]}
    lists = []
    for k in ["changed","changes","diffs","block_diffs","mutations"]:
        if isinstance(obj.get(k), list):
            lists.extend(obj[k])
    if not lists and any(k in obj for k in ["block","block_id","old","new","mutation"]):
        lists = [obj]
    for d in lists:
        if not isinstance(d, dict):
            continue
        row = dict(base)
        row.update({
            "block": d.get("block", d.get("block_id", d.get("target_block_id"))),
            "mutation": d.get("mutation", d.get("operator", d.get("mutation_type"))),
            "old": d.get("old"),
            "new": d.get("new"),
            "source": d.get("source"),
            "compression_level": d.get("compression_level"),
        })
        row["case_label_inferred"] = classify_case_abc(row)
        rows.append(row)
    return rows

diff_rows = []
for fn in ["ga_block_diffs.jsonl", "advisor_accepted_block_diffs.jsonl"]:
    for obj in read_jsonl(OUT_DIR/fn):
        diff_rows.extend(flatten_block_diff(obj))
block_diff_df = pd.DataFrame(diff_rows)

print("\n[BLOCK DIFFS]")
display(block_diff_df if len(block_diff_df) else pd.DataFrame([{"message":"No block diffs parsed"}]))

print("\n[PROMPT BEFORE/AFTER FILES]")
for p in sorted(OUT_DIR.glob("advisor_prompt_before_after_generation_*.json"))[:10]:
    obj = read_json(p) or {}
    print(p.name, {k: obj.get(k) for k in ["parent_prompt_tokens","child_prompt_tokens","token_delta","changed_blocks","removed_blocks","changed_few_shot_count","changed_max_tokens"] if k in obj})

In [ ]:
# ============================================================
# Cell 14. Prompt token and DETPass trend
# ============================================================

OUT_DIR = select_advisor_out_dir()
progress = load_progress(OUT_DIR)
trans = load_transitions(OUT_DIR)

if len(progress):
    cols = [c for c in ["generation","validation_det_pass_rate","validation_avg_det_score","best_so_far_DETPass","avg_prompt_tokens","compression_ready","compression_phase"] if c in progress.columns]
    display(progress[cols])
    try:
        import matplotlib.pyplot as plt
        if "generation" in progress.columns and "avg_prompt_tokens" in progress.columns:
            plt.figure()
            plt.plot(progress["generation"], pd.to_numeric(progress["avg_prompt_tokens"], errors="coerce"), marker="o")
            plt.xlabel("Generation")
            plt.ylabel("Average prompt tokens")
            plt.title("Prompt token trend")
            plt.grid(True, alpha=0.3)
            plt.show()
        det_col = "best_so_far_DETPass" if "best_so_far_DETPass" in progress.columns else "validation_det_pass_rate"
        if "generation" in progress.columns and det_col in progress.columns:
            plt.figure()
            plt.plot(progress["generation"], pd.to_numeric(progress[det_col], errors="coerce"), marker="o")
            plt.xlabel("Generation")
            plt.ylabel(det_col)
            plt.title("DETPass trend")
            plt.grid(True, alpha=0.3)
            plt.show()
    except Exception as e:
        print("[WARN] plot failed:", repr(e))
else:
    print("[WARN] no progress")

In [ ]:
# ============================================================
# Cell 15. Case A/B/C summary
# ============================================================

OUT_DIR = select_advisor_out_dir()
case_path = OUT_DIR / "advisor_case_ABC_summary.csv"
if case_path.exists():
    case_summary_df = pd.read_csv(case_path)
else:
    prop = load_all_proposals(OUT_DIR)
    rows = []
    if len(prop):
        for case, g in prop.groupby("case_label_inferred", dropna=False):
            rows.append({
                "case_label": case,
                "proposals_generated": len(g),
                "proposals_accepted": int((g.get("accepted", pd.Series(False, index=g.index)) == True).sum()) if "accepted" in g.columns else np.nan,
                "proposals_rejected": int((g.get("accepted", pd.Series(False, index=g.index)) == False).sum()) if "accepted" in g.columns else np.nan,
                "mean_expected_token_delta": pd.to_numeric(g.get("expected_token_delta"), errors="coerce").mean() if "expected_token_delta" in g.columns else np.nan,
                "mean_measured_token_delta": pd.to_numeric(g.get("measured_prompt_token_delta"), errors="coerce").mean() if "measured_prompt_token_delta" in g.columns else np.nan,
                "most_common_operator": g["operator"].mode().iloc[0] if "operator" in g.columns and len(g["operator"].dropna()) else None,
            })
    case_summary_df = pd.DataFrame(rows)
display(case_summary_df if len(case_summary_df) else pd.DataFrame([{"message":"No Case A/B/C summary"}]))

In [ ]:
# ============================================================
# Cell 16. Three-way comparison
# ============================================================

runs = []
if "cloudless_out" in globals(): runs.append(("cloudless", cloudless_out))
if "hybrid_out" in globals(): runs.append(("hybrid_advisor", hybrid_out))
if "cloud_only_out" in globals(): runs.append(("cloud_only_advisor", cloud_only_out))

if runs:
    three_way_df = pd.DataFrame([summarize_run(out, label=label, mode=label) for label, out in runs])
    display(three_way_df)
    out_csv = SUMMARY_DIR / f"cloudless_vs_hybrid_vs_cloudonly_{SERVER_PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    three_way_df.to_csv(out_csv, index=False)
    print("saved:", out_csv)
else:
    print("[WARN] no run variables available")

In [ ]:
# ============================================================
# Cell 17. Final gates
# ============================================================

def gate_common(out_dir, threshold=90):
    errors, warnings = [], []
    out_dir = Path(out_dir)
    p = load_progress(out_dir)
    t = load_transitions(out_dir)
    for fn in ["ga_summary.json","ga_generation_progress.csv","population_transitions.csv"]:
        if not (out_dir/fn).exists(): errors.append(f"missing {fn}")
    if len(p):
        if "avg_prompt_tokens" not in p.columns or pd.to_numeric(p["avg_prompt_tokens"], errors="coerce").fillna(0).max() <= 0:
            errors.append("avg_prompt_tokens missing or non-positive")
        dets = []
        for c in ["best_so_far_DETPass","validation_det_pass_rate","det_pass_rate"]:
            if c in p.columns:
                dets.append(float(pd.to_numeric(p[c], errors="coerce").fillna(0).max()))
        if max(dets) if dets else 0 < threshold:
            errors.append(f"DETPass below threshold {threshold}")
    else:
        errors.append("progress empty")
    return errors, warnings

def gate_cloud_only(out_dir):
    errors, warnings = gate_common(out_dir)
    out_dir = Path(out_dir)
    prop = load_all_proposals(out_dir)
    trans = load_transitions(out_dir)
    if len(prop) and "source" in prop.columns:
        fb = int((prop["source"].astype(str) == "compression_fallback").sum())
        if fb: errors.append(f"fallback rows exist in cloud-only run: {fb}")
    if len(trans) and "new_by_compression_fallback" in trans.columns:
        fb_sum = float(pd.to_numeric(trans["new_by_compression_fallback"], errors="coerce").fillna(0).sum())
        if fb_sum != 0: errors.append(f"new_by_compression_fallback_sum={fb_sum}")
    if not (out_dir/"advisor_cloud_only_before_after.csv").exists():
        errors.append("missing advisor_cloud_only_before_after.csv")
    if not (out_dir/"advisor_case_ABC_summary.csv").exists():
        errors.append("missing advisor_case_ABC_summary.csv")
    advisor_rows = int((prop.get("_file", pd.Series(dtype=str)) == "advisor_mutation_proposals.jsonl").sum()) if len(prop) else 0
    if advisor_rows <= 0:
        errors.append("advisor proposal rows == 0")
    accepted = int((prop.get("accepted", pd.Series(dtype=bool)) == True).sum()) if len(prop) and "accepted" in prop.columns else 0
    applied = int((prop.get("applied", pd.Series(dtype=bool)) == True).sum()) if len(prop) and "applied" in prop.columns else 0
    unfilled = float(pd.to_numeric(trans.get("advisor_unfilled_quota", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()) if len(trans) else 0
    if accepted + applied <= 0 and unfilled <= 0:
        errors.append("no accepted/applied advisor proposal and no advisor_unfilled_quota")
    return errors, warnings

targets = []
if "cloudless_out" in globals(): targets.append(("cloudless", cloudless_out, gate_common))
if "hybrid_out" in globals(): targets.append(("hybrid", hybrid_out, gate_common))
if "cloud_only_out" in globals(): targets.append(("cloud_only", cloud_only_out, gate_cloud_only))

if not targets:
    print("[WARN] no run variables found")
else:
    for name, out, fn in targets:
        print("\n" + "="*120)
        print(f"[FINAL GATE: {name}]")
        errors, warnings = fn(out)
        if warnings:
            print("[WARNINGS]"); [print("-", w) for w in warnings]
        if errors:
            print("[DO NOT PROMOTE]"); [print("-", e) for e in errors]
        else:
            print("[OK]", name, "gate passed")

In [ ]:
# ============================================================
# Cell 18. Optional full run command cells
# Disabled by default.
# ============================================================

FULL_COMMON = dict(
    categories=tuple(range(1, 9)),
    limit_per_category=3,
    sample_size=24,
    validation_size=24,
    population=5,
    gens=10,
    target_detpass=90,
    full_run=True,
    progress="verbose",
    timeout_sec=3600,
    retries=0,
)

def run_full_cloud_only(label, model_key):
    return run_ga_all_categories(
        model_key=model_key,
        base_prefix=f"ga_{SERVER_PRESET}_cloud_only_advisor",
        use_advisor=True,
        advisor_model_key="gpt41_mini",
        advisor_trigger_mode="always",
        advisor_min_population_for_child=4,
        advisor_force_child_quota=True,
        cloud_only_advisor_compression=True,
        disable_compression_fallback=True,
        compression_child_quota=0,
        compression_child_ratio=0.0,
        micro_compression_child_quota=0,
        micro_compression_child_ratio=0.0,
        block_compression_child_quota=0,
        block_compression_child_ratio=0.0,
        multi_block_compression_child_quota=0,
        multi_block_compression_child_ratio=0.0,
        global_budget_compression_child_quota=0,
        advisor_compression_child_quota=2,
        advisor_proposal_k=6,
        advisor_min_usable_compression_proposals=2,
        advisor_repair_invalid_proposals=True,
        advisor_repair_max_attempts=2,
        advisor_require_block_proposal_after_detpass=True,
        advisor_require_token_delta=True,
        advisor_disallow_genome_only_after_detpass=True,
        advisor_cloud_only_strict_schema=True,
        advisor_before_after_report=True,
        advisor_include_dataset_feedback=True,
        advisor_feedback_topk=5,
        advisor_feedback_max_failures_per_family=3,
        advisor_include_prompt_before_after_context=True,
        advisor_include_case_abc_sections=True,
        advisor_write_debug_prompt_sections=True,
        **FULL_COMMON,
    )

if RUN_FULL_CLOUD_ONLY:
    full_cloud_only_runs = {}
    for label, model_key in MODEL_LIST_3:
        full_cloud_only_runs[label] = run_full_cloud_only(label, model_key)
else:
    print("[SKIP] RUN_FULL_CLOUD_ONLY=False")

if RUN_FULL_THREE_WAY:
    full_three_way_runs = {}
    for label, model_key in MODEL_LIST_3:
        full_three_way_runs[f"{label}_cloudless"] = run_ga_all_categories(model_key=model_key, base_prefix=f"ga_{SERVER_PRESET}_cloudless", use_advisor=False, **FULL_COMMON)
        full_three_way_runs[f"{label}_hybrid"] = run_ga_all_categories(model_key=model_key, base_prefix=f"ga_{SERVER_PRESET}_hybrid", use_advisor=True, advisor_model_key="gpt41_mini", **FULL_COMMON)
        full_three_way_runs[f"{label}_cloud_only"] = run_full_cloud_only(label, model_key)
else:
    print("[SKIP] RUN_FULL_THREE_WAY=False")

In [ ]:
# ============================================================
# Cell 19. Save paper artifacts
# ============================================================

def save_prompt_sections_md(out_dir, output_path):
    out_dir = Path(out_dir)
    lines = []
    for p in sorted(out_dir.glob("advisor_prompt_generation_*.txt")):
        txt = p.read_text(encoding="utf-8", errors="replace")
        lines.append(f"# {p.name}\n")
        for s in ["CURRENT_STATE","DATASET_EVALUATION_FEEDBACK","CASE_A_MICRO_COMPRESSION","CASE_B_THRESHOLD_BLOCK_COMPRESSION","CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION","STRICT_JSON_RESPONSE_SCHEMA"]:
            sec = extract_section_text(txt, s, 3500) if "extract_section_text" in globals() else ""
            if sec:
                lines.append(f"## {s}\n\n```text\n{sec}\n```\n")
    Path(output_path).write_text("\n".join(lines), encoding="utf-8")

saved = []
runs = []
if "cloudless_out" in globals(): runs.append(("cloudless", cloudless_out))
if "hybrid_out" in globals(): runs.append(("hybrid_advisor", hybrid_out))
if "cloud_only_out" in globals(): runs.append(("cloud_only_advisor", cloud_only_out))

if runs:
    main_df = pd.DataFrame([summarize_run(out, label=label, mode=label) for label, out in runs])
    display(main_df)
    p = TABLE_DIR / f"cloudless_vs_hybrid_vs_cloudonly_{SERVER_PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    main_df.to_csv(p, index=False); saved.append(p)

if "cloud_only_out" in globals():
    out = Path(cloud_only_out)
    for src, prefix in [("advisor_cloud_only_before_after.csv","cloud_only_before_after"),("advisor_case_ABC_summary.csv","cloud_only_case_ABC_summary")]:
        sp = out/src
        if sp.exists():
            dp = TABLE_DIR / f"{prefix}_{SERVER_PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            pd.read_csv(sp).to_csv(dp, index=False); saved.append(dp)
    mdp = TABLE_DIR / f"advisor_prompt_sections_{SERVER_PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
    save_prompt_sections_md(out, mdp); saved.append(mdp)

print("[SAVED]")
for p in saved:
    print(p)

In [ ]:
# ============================================================
# Cell 20. Interpretation template
# ============================================================

def build_interpretation(out_dir, mode_label):
    row = summarize_run(out_dir, label=mode_label, mode=mode_label)
    prop = load_all_proposals(out_dir)
    fallback_rows = row.get("fallback_rows", 0)
    case_counts = {}
    if len(prop):
        case_counts = prop.groupby("case_label_inferred").size().to_dict()

    kr = (
        f"[한국어 해석 템플릿]\n\n"
        f"본 실행은 {mode_label} 모드에서 수행되었다.\n"
        f"dataset evaluation 결과는 DATASET_EVALUATION_FEEDBACK 섹션으로 cloud advisor prompt에 전달된다.\n"
        f"이 섹션에는 DETPass, validation pass rate, token trend, failure family, target block mapping, "
        f"prompt_token_breakdown, block_token_breakdown 정보가 포함된다.\n\n"
        f"실행 결과:\n"
        f"- best_so_far_DETPass: {row.get('best_so_far_DETPass', row.get('validation_det_pass_rate'))}\n"
        f"- first prompt tokens: {row.get('first_tokens')}\n"
        f"- minimum prompt tokens: {row.get('min_tokens')}\n"
        f"- best token reduction ratio: {row.get('token_reduction_ratio_min')}\n"
        f"- advisor accepted rows: {row.get('advisor_accepted_rows')}\n"
        f"- advisor rejected rows: {row.get('advisor_rejected_rows')}\n"
        f"- fallback rows: {fallback_rows}\n"
        f"- Case A/B/C proposal counts: {case_counts}\n\n"
        f"해석 기준:\n"
        f"- fallback rows가 0이면 cloud-only advisor isolation 조건을 만족한다.\n"
        f"- fallback rows가 0보다 크면 advisor-only 기여로 주장하면 안 된다.\n"
        f"- token 감소는 measured_prompt_token_delta 또는 before/after artifact가 있을 때만 advisor 기여로 주장한다.\n"
        f"- accepted/applied proposal의 before-after block/genome/token diff가 있어야 cloud advisor 기반 prompt compression이라고 볼 수 있다.\n"
    )

    en = (
        f"[English interpretation template]\n\n"
        f"This run was executed in {mode_label} mode.\n"
        f"Dataset evaluation feedback is injected into the cloud advisor prompt through DATASET_EVALUATION_FEEDBACK.\n"
        f"The section includes DETPass, validation pass rate, token trend, failure families, target block mapping, "
        f"prompt_token_breakdown, and block_token_breakdown.\n\n"
        f"Observed results:\n"
        f"- best_so_far_DETPass: {row.get('best_so_far_DETPass', row.get('validation_det_pass_rate'))}\n"
        f"- first prompt tokens: {row.get('first_tokens')}\n"
        f"- minimum prompt tokens: {row.get('min_tokens')}\n"
        f"- best token reduction ratio: {row.get('token_reduction_ratio_min')}\n"
        f"- advisor accepted rows: {row.get('advisor_accepted_rows')}\n"
        f"- advisor rejected rows: {row.get('advisor_rejected_rows')}\n"
        f"- fallback rows: {fallback_rows}\n"
        f"- Case A/B/C proposal counts: {case_counts}\n\n"
        f"Interpretation criteria:\n"
        f"- fallback rows equal to zero means cloud-only advisor isolation is satisfied.\n"
        f"- nonzero fallback rows mean compression includes fallback contribution, not advisor-only contribution.\n"
        f"- token reduction should be attributed to advisor only when measured_prompt_token_delta or before/after artifacts exist.\n"
        f"- accepted/applied proposal-level before-after block/genome/token diff is required to claim cloud-advisor-driven compression.\n"
    )
    return kr, en

if "cloud_only_out" in globals():
    kr, en = build_interpretation(cloud_only_out, "cloud-only advisor")
    print(kr); print("\n" + "="*120 + "\n"); print(en)
elif "hybrid_out" in globals():
    kr, en = build_interpretation(hybrid_out, "hybrid advisor")
    print(kr); print("\n" + "="*120 + "\n"); print(en)
else:
    print("[WARN] Run cloud-only or hybrid smoke first.")